# 06 Email and web features: do they lift the strong tabular baselines?
Pre-registered in `docs/PROJECT_LOG.md` section 6b (written before any model was fit on these features): feature sets, decision rule, interpretation rule.
Repeat 0 only (5 folds), RF and XGBoost with the fixed notebook-04 hyperparameters, no tuning, shared `make_folds` folds, split by user.
A GO would mean the email/web logs carry signal, not that a graph helps.

In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from itgnn.commslogs import add_comms_features, comms_feature_sets, NEW_BASE
from itgnn.data import load_config, prepare
from itgnn.evaluation import alert_budget, evaluate_scores, make_folds
from itgnn.weekly_features import FEATURE_SETS, add_history_features, drop_warmup
from itgnn.windows import label_weeks, load_attack_windows, make_weekly_features, week_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml")
SEED, QS = cfg["seed"], [0.25, 0.5, 1, 2]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a")
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 60
pd.options.display.max_rows = 200

d = prepare(cfg)
users, labels = d["users"], d["labels"]
folds = [f for f in make_folds(users, labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED) if f["repeat"] == 0]
assert len(folds) == 5
windows = load_attack_windows(cfg["raw_dir"])
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]
SCEN = sorted(scen.unique())

## 1. Weekly rows, features, and checks against the saved W2 run

In [2]:
full = make_weekly_features(d["logon"], d["device"], d["file"], users)
full["label"] = label_weeks(full, windows)
full["cal_week"] = (full["week_start"] - full["week_start"].min()).dt.days // 7
full = add_history_features(full, d["logon"])           # sorted by (user, week_start), W2 history, before the warm-up drop
email = pd.read_parquet("data/processed/email.parquet")
agg = pd.read_parquet("data/processed/http_user_week_domain.parquet")
full, dropped = add_comms_features(full, email, agg)
weekly = drop_warmup(full)
y = weekly["label"]
print(f"rows {len(full):,} -> {len(weekly):,} after warm-up; attack weeks {int(weekly['label'].sum())}; prevalence {y.mean():.4f}")
print("user-weeks with email / web events but NO Logon row (dropped from the model):", dropped)
print("email events in the dropped weeks are counted as user-weeks, not events.")

saved = pd.read_parquet("results/weekly_v2_oof_scores.parquet", columns=["user", "week_start", "model", "repeat", "fold", "score"])
keys_saved = saved[saved["model"] == "RF W2"].drop_duplicates(["user", "week_start"])[["user", "week_start"]]
k1 = weekly[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
k2 = keys_saved.sort_values(["user", "week_start"]).reset_index(drop=True)
pd.testing.assert_frame_equal(k1, k2)
print("row keys identical to the saved W2 run:", len(k1), "user-weeks")
SETS = comms_feature_sets(FEATURE_SETS["W2"])
allcols = sorted({c for v in SETS.values() for c in v})
assert np.isfinite(weekly[allcols].to_numpy(float)).all(), "NaN or inf in features"
print({k: len(v) for k, v in SETS.items()})

rows 67,238 -> 63,238 after warm-up; attack weeks 322; prevalence 0.0051
user-weeks with email / web events but NO Logon row (dropped from the model): {'email': 0, 'web': 0}
email events in the dropped weeks are counted as user-weeks, not events.
row keys identical to the saved W2 run: 63238 user-weeks


{'W2': 16, 'W2+E': 49, 'W2+H': 46, 'W2+E+H': 79, 'W2+E+H generic': 61, 'W2+E+H no-first-seen': 70, 'W2+calendar': 17}


In [3]:
# descriptive sanity checks on the new features (nothing is selected from these)
cov = weekly.groupby(weekly["user"].isin(labels).map({True: "insiders", False: "normal"}))[["em_n", "wb_n"]].agg(lambda s: (s > 0).mean())
print("share of user-weeks with any email / any web activity, by group:"); display(cov)
print("wb_share_rare, mean by calendar-week block (burn-in: domain history is nearly empty at first):")
display(weekly.groupby(pd.cut(weekly["cal_week"], [-1, 3, 7, 12, 25, 50, 80]), observed=True)["wb_share_rare"].agg(["mean", "size"]))
print("category counts: user-weeks with a visit, and distinct users with a visit")
rows = {c: {"user-weeks": int((weekly[c] > 0).sum()), "users": weekly.loc[weekly[c] > 0, "user"].nunique()} for c in [c for c in NEW_BASE if c.startswith("wb_n_") and c.split("wb_n_")[1] in ("leak", "job", "cloud", "download", "competitor")] + ["em_n_competitor_recip"]}
display(pd.DataFrame(rows).T)

share of user-weeks with any email / any web activity, by group:


,em_n,wb_n
user,,
insiders,1.000,1.000
normal,1.000,1.000


wb_share_rare, mean by calendar-week block (burn-in: domain history is nearly empty at first):


,mean,size
cal_week,,
"(3, 7]",0.000,3065
"(7, 12]",0.000,4939
"(12, 25]",0.000,12705
"(25, 50]",0.000,23443
"(50, 80]",0.000,19086


category counts: user-weeks with a visit, and distinct users with a visit


,user-weeks,users
wb_n_leak,52,30
wb_n_job,33942,902
wb_n_cloud,31104,849
wb_n_download,9,9
wb_n_competitor,18942,513
em_n_competitor_recip,51919,999


## 2. Fits (7 sets x RF/XGBoost x 5 folds = 70). Same hyperparameters as notebook 04

In [4]:
def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
def xgb(ytr):
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(),
                         n_jobs=4, random_state=SEED, verbosity=0)
ALGOS = {"RF": lambda ytr: rf(), "XGB": xgb}
t0 = time.time(); metric_rows, oof_parts, fit_times = [], [], []
for f in folds:
    tr, te = week_masks(weekly, f)
    for sname, cols in SETS.items():
        Xtr, Xte = weekly.loc[tr, cols], weekly.loc[te, cols]
        for a, mk in ALGOS.items():
            ts = time.time()
            s = mk(y[tr]).fit(Xtr, y[tr]).predict_proba(Xte)[:, 1]
            assert np.isfinite(s).all()
            ev = evaluate_scores(y[te].to_numpy(), s, k=100)
            name = f"{a} {sname}"
            metric_rows.append({"model": name, "algo": a, "set": sname, "repeat": f["repeat"], "fold": f["fold"], **ev})
            fit_times.append(time.time() - ts)
            oof_parts.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(),
                                           "cal_week": weekly.loc[te, "cal_week"].to_numpy(), "repeat": f["repeat"], "fold": f["fold"],
                                           "model": name, "score": s, "label": y[te].to_numpy()}))
    print(f"fold {f['fold']} done, elapsed {time.time() - t0:.0f}s", file=LOG, flush=True)
results = pd.DataFrame(metric_rows)
oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet("results/comms_probe_oof_scores.parquet", index=False)
print(f"70 fits in {time.time() - t0:.0f}s (mean fit {np.mean(fit_times):.1f}s, max {np.max(fit_times):.1f}s); saved results/comms_probe_oof_scores.parquet {oof.shape}")

70 fits in 158s (mean fit 2.2s, max 5.7s); saved results/comms_probe_oof_scores.parquet (885332, 8)


## 3. Reproduction check: W2 on repeat 0 must match the saved step-4 scores

In [5]:
rep = []
for a in ["RF", "XGB"]:
    sv = saved[(saved["model"] == f"{a} W2") & (saved["repeat"] == 0)]
    for fo, g in oof[oof["model"] == f"{a} W2"].groupby("fold"):
        m = g.merge(sv[sv["fold"] == fo], on=["user", "week_start"], suffixes=("", "_saved"))
        assert len(m) == len(g)
        new = evaluate_scores(m["label"], m["score"], 100)["pr_auc"]; old = evaluate_scores(m["label"], m["score_saved"], 100)["pr_auc"]
        rep.append({"algo": a, "fold": fo, "pr_auc now": new, "pr_auc saved": old, "abs diff": abs(new - old), "max score diff": (m["score"] - m["score_saved"]).abs().max()})
rep = pd.DataFrame(rep); display(rep)
print("max PR-AUC diff RF:", rep.loc[rep.algo == "RF", "abs diff"].max(), "| XGB:", rep.loc[rep.algo == "XGB", "abs diff"].max())

,algo,fold,pr_auc now,pr_auc saved,abs diff,max score diff
0,RF,0,0.463,0.463,0.000,0.000
1,RF,1,0.626,0.626,0.000,0.000
2,RF,2,0.468,0.468,0.000,0.000
3,RF,3,0.477,0.477,0.000,0.000
4,RF,4,0.460,0.460,0.000,0.000
5,XGB,0,0.435,0.435,0.000,0.000
6,XGB,1,0.617,0.617,0.000,0.000
7,XGB,2,0.414,0.414,0.000,0.000
8,XGB,3,0.497,0.497,0.000,0.000
9,XGB,4,0.362,0.362,0.000,0.000


max PR-AUC diff RF: 0.0 | XGB: 0.0


## 4. PR-AUC and paired differences vs W2 (same folds)

In [6]:
S = results.groupby("model")[["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]].agg(["mean", "std"])
order = [f"{a} {s}" for a in ["RF", "XGB"] for s in SETS]
display(pd.DataFrame({m: S[m].apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1) for m in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]}).loc[order])
pv = results.pivot_table(index="fold", columns="model", values="pr_auc")
print("per-fold PR-AUC:"); display(pv[order])
rows = {}
for a in ["RF", "XGB"]:
    for s in list(SETS)[1:]:
        dl = pv[f"{a} {s}"] - pv[f"{a} W2"]
        rows[f"{a}: {s} - W2"] = {"mean diff": dl.mean(), "std": dl.std(), "folds positive": f"{int((dl > 0).sum())}/5", **{f"fold {i}": dl[i] for i in dl.index}}
paired = pd.DataFrame(rows).T; display(paired)

,pr_auc,roc_auc,precision_at_k,recall_at_k
model,,,,
RF W2,0.499 ± 0.072,0.979 ± 0.018,0.352 ± 0.022,0.551 ± 0.073
RF W2+E,0.575 ± 0.122,0.975 ± 0.023,0.408 ± 0.081,0.640 ± 0.152
RF W2+H,0.772 ± 0.101,0.998 ± 0.002,0.544 ± 0.053,0.851 ± 0.126
RF W2+E+H,0.743 ± 0.098,0.997 ± 0.002,0.516 ± 0.078,0.807 ± 0.148
RF W2+E+H generic,0.675 ± 0.127,0.987 ± 0.011,0.450 ± 0.086,0.707 ± 0.171
RF W2+E+H no-first-seen,0.709 ± 0.094,0.996 ± 0.003,0.494 ± 0.080,0.773 ± 0.155
RF W2+calendar,0.590 ± 0.105,0.986 ± 0.015,0.420 ± 0.058,0.661 ± 0.148
XGB W2,0.465 ± 0.097,0.986 ± 0.007,0.346 ± 0.030,0.542 ± 0.082
XGB W2+E,0.566 ± 0.107,0.987 ± 0.008,0.406 ± 0.049,0.636 ± 0.107


per-fold PR-AUC:


model,RF W2,RF W2+E,RF W2+H,RF W2+E+H,RF W2+E+H generic,RF W2+E+H no-first-seen,RF W2+calendar,XGB W2,XGB W2+E,XGB W2+H,XGB W2+E+H,XGB W2+E+H generic,XGB W2+E+H no-first-seen,XGB W2+calendar
fold,,,,,,,,,,,,,,
0,0.463,0.594,0.744,0.741,0.700,0.679,0.521,0.435,0.587,0.834,0.841,0.788,0.846,0.515
1,0.626,0.728,0.776,0.833,0.779,0.834,0.765,0.617,0.719,0.904,0.928,0.916,0.947,0.741
2,0.468,0.644,0.935,0.773,0.774,0.723,0.600,0.414,0.596,0.984,0.979,0.939,0.979,0.497
3,0.477,0.420,0.659,0.579,0.468,0.574,0.563,0.497,0.474,0.783,0.769,0.658,0.764,0.615
4,0.460,0.489,0.746,0.787,0.654,0.736,0.500,0.362,0.453,0.870,0.878,0.819,0.850,0.456


,mean diff,std,folds positive,fold 0,fold 1,fold 2,fold 3,fold 4
RF: W2+E - W2,0.076,0.091,4/5,0.130,0.101,0.176,-0.057,0.029
RF: W2+H - W2,0.273,0.124,5/5,0.280,0.150,0.467,0.182,0.286
RF: W2+E+H - W2,0.244,0.091,5/5,0.278,0.207,0.305,0.102,0.327
RF: W2+E+H generic - W2,0.176,0.118,4/5,0.237,0.152,0.306,-0.009,0.194
RF: W2+E+H no-first-seen - W2,0.210,0.069,5/5,0.216,0.207,0.255,0.097,0.276
RF: W2+calendar - W2,0.091,0.044,5/5,0.058,0.139,0.132,0.086,0.040
XGB: W2+E - W2,0.101,0.078,4/5,0.152,0.102,0.181,-0.022,0.090
XGB: W2+H - W2,0.410,0.128,5/5,0.399,0.288,0.569,0.287,0.508
XGB: W2+E+H - W2,0.414,0.126,5/5,0.407,0.311,0.564,0.272,0.515
XGB: W2+E+H generic - W2,0.359,0.141,5/5,0.353,0.299,0.524,0.162,0.457


## 5. Alert budget and per scenario (repeat 0, one repeat so no spread)

In [7]:
scen_of = scen.to_dict()
bud = alert_budget(oof, labels, scen_of, QS, SEED)
elig = weekly[weekly["label"] == 1].groupby("user").size().index
print("eligible insiders per scenario:", {int(s): int((scen[list(elig)] == s).sum()) for s in SCEN}, "| normal users:", int(weekly.loc[~weekly["user"].isin(labels), "user"].nunique()))
cols = ["caught", "normal flagged"] + [f"caught s{s}" for s in SCEN]
b = bud.set_index(["model", "q%"])[cols]
for a in ["RF", "XGB"]:
    print(f"--- {a}: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario"); display(b.loc[[f"{a} {s}" for s in SETS]])

eligible insiders per scenario: {1: 30, 2: 30, 3: 10} | normal users: 930
--- RF: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario


caught  normal flagged  caught s1  caught s2  caught s3
model                   q%                                                            
RF W2                   0.250      49               3         22         23          4
                        0.500      54              10         26         24          4
                        1.000      60              19         28         26          6
                        2.000      65              41         29         29          7
RF W2+E                 0.250      43               2         19         20          4
                        0.500      54               8         26         22          6
                        1.000      57              17         28         23          6
                        2.000      60              46         28         25          7
RF W2+H                 0.250      39               1         17         20          2
                        0.500      60               8         26         28          6
                        1.000      67              26         30         29          8
                        2.000      70              79         30         30         10
RF W2+E+H               0.250      41               2         19         19          3
                        0.500      58               6         28         24          6
                        1.000      66              20         30         29          7
                        2.000      70              67         30         30         10
RF W2+E+H generic       0.250      44               3         18         20          6
                        0.500      54               7         24         23          7
                        1.000      60              23         28         24          8
                        2.000      65              54         29         28          8
RF W2+E+H no-first-seen 0.250      38               2         17         18          3
                        0.500      59               6         28         26          5
                        1.000      65              23         30         28          7
                        2.000      67              56         30         29          8
RF W2+calendar          0.250      45               2         19         23          3
                        0.500      52               8         25         24          3
                        1.000      60              29         28         27          5
                        2.000      67              73         30         30          7

--- XGB: insiders caught (of 70 eligible), normal users flagged (of 930), caught per scenario


caught  normal flagged  caught s1  caught s2  caught s3
model                    q%                                                            
XGB W2                   0.250      51               2         22         25          4
                         0.500      53               5         24         25          4
                         1.000      58              12         27         27          4
                         2.000      62              36         29         29          4
XGB W2+E                 0.250      41               1         16         22          3
                         0.500      52               8         24         24          4
                         1.000      57              23         27         26          4
                         2.000      62              44         29         29          4
XGB W2+H                 0.250      56               1         30         24          2
                         0.500      62               6         30         29          3
                         1.000      67              26         30         29          8
                         2.000      69              62         30         30          9
XGB W2+E+H               0.250      56               0         30         24          2
                         0.500      62               7         30         28          4
                         1.000      68              27         30         30          8
                         2.000      69              51         30         30          9
XGB W2+E+H generic       0.250      50               1         20         26          4
                         0.500      59              11         27         28          4
                         1.000      66              23         30         29          7
                         2.000      67              63         30         29          8
XGB W2+E+H no-first-seen 0.250      54               0         28         22          4
                         0.500      63               9         30         29          4
                         1.000      66              24         30         29          7
                         2.000      68              49         30         30          8
XGB W2+calendar          0.250      44               2         18         24          2
                         0.500      50               7         22         25          3
                         1.000      56              26         25         28          3
                         2.000      62              64         29         29          4

## 6. Burn-in check (descriptive): PR-AUC restricted to calendar week >= 13, from the saved predictions

In [8]:
bi = []
for (m, fo), g in oof.groupby(["model", "fold"]):
    g = g[g["cal_week"] >= 13]
    bi.append({"model": m, "fold": fo, "pr_auc_w13": evaluate_scores(g["label"], g["score"], 100)["pr_auc"]})
bi = pd.DataFrame(bi); pb = bi.pivot_table(index="fold", columns="model", values="pr_auc_w13")
print("rows kept: weeks >= 13;  attack weeks:", int(oof[(oof['model'] == 'RF W2') & (oof['cal_week'] >= 13)]['label'].sum()), "of", int(oof[oof['model'] == 'RF W2']['label'].sum()))
t = pd.DataFrame({"PR-AUC wk>=13 (mean ± std)": pb[order].apply(lambda s: f"{s.mean():.3f} ± {s.std():.3f}")})
for a in ["RF", "XGB"]:
    for s in list(SETS)[1:]:
        t.loc[f"{a} {s}", "paired diff vs W2"] = (pb[f"{a} {s}"] - pb[f"{a} W2"]).mean()
display(t.loc[order])

rows kept: weeks >= 13;  attack weeks: 322 of 322


,PR-AUC wk>=13 (mean ± std),paired diff vs W2
model,,
RF W2,0.527 ± 0.079,NaN
RF W2+E,0.587 ± 0.127,0.059
RF W2+H,0.772 ± 0.101,0.245
RF W2+E+H,0.743 ± 0.097,0.216
RF W2+E+H generic,0.679 ± 0.127,0.152
RF W2+E+H no-first-seen,0.711 ± 0.094,0.183
RF W2+calendar,0.590 ± 0.105,0.063
XGB W2,0.500 ± 0.099,NaN
XGB W2+E,0.579 ± 0.115,0.079


## 7. Pre-registered decision rule (docs/PROJECT_LOG.md 6b)

In [9]:
def dl(a, s): return pv[f"{a} {s}"] - pv[f"{a} W2"]
rf_d, xg_d = dl("RF", "W2+E+H"), dl("XGB", "W2+E+H")
q5 = bud[bud["q%"] == 0.5].set_index("model")
dc = q5.loc["RF W2+E+H", "caught"] - q5.loc["RF W2", "caught"]; dn = q5.loc["RF W2+E+H", "normal flagged"] - q5.loc["RF W2", "normal flagged"]
cal = dl("RF", "W2+calendar").mean()
crit = {
 "1. RF mean delta PR-AUC >= +0.05": (rf_d.mean(), rf_d.mean() >= 0.05),
 "2. RF >= 4 of 5 folds positive": (int((rf_d > 0).sum()), (rf_d > 0).sum() >= 4),
 "3. XGB mean delta >= +0.03 and >= 4/5 folds positive": (f"{xg_d.mean():.3f}, {int((xg_d > 0).sum())}/5", xg_d.mean() >= 0.03 and (xg_d > 0).sum() >= 4),
 "4. q=0.5%: RF +>=3 insiders caught, <=2 extra normal flagged": (f"+{dc:g} caught, {dn:+g} normal", dc >= 3 and dn <= 2),
 "5. calendar control delta < +0.01": (cal, cal < 0.01),
}
display(pd.DataFrame(crit, index=["value", "met"]).T)
if all(v[1] for v in crit.values()): verdict = "GO"
elif rf_d.mean() < 0.02: verdict = "NO-GO"
else: verdict = "AMBIGUOUS"
print("VERDICT:", verdict, "| RF mean delta", round(rf_d.mean(), 4), "| XGB mean delta", round(xg_d.mean(), 4))
gen = dl("RF", "W2+E+H generic").mean()
print("generic RF mean delta vs W2:", round(gen, 4),
      "->", "named-site only (generic < +0.02): do NOT build a graph" if gen < 0.02 else "case for a relational graph (generic >= +0.03)" if gen >= 0.03 else "grey zone (0.02-0.03), no conclusion")
top = results[results["model"] == "RF W2+E+H"]["pr_auc"].mean()
print("W2+E+H RF PR-AUC", round(top, 3), "-> leakage audit REQUIRED" if top > 0.80 else "-> below the 0.80 leakage-audit trigger")
print("NOTE: wikileaks.org has exactly 30 users and scenario 1 has 30 insiders; a scenario-1 gain is expected to be easy here.")

,value,met
1. RF mean delta PR-AUC >= +0.05,0.244,True
2. RF >= 4 of 5 folds positive,5,True
3. XGB mean delta >= +0.03 and >= 4/5 folds positive,"0.414, 5/5",True
"4. q=0.5%: RF +>=3 insiders caught, <=2 extra normal flagged","+4 caught, -4 normal",True
5. calendar control delta < +0.01,0.091,False


VERDICT: AMBIGUOUS | RF mean delta 0.2436 | XGB mean delta 0.4139
generic RF mean delta vs W2: 0.1757 -> case for a relational graph (generic >= +0.03)
W2+E+H RF PR-AUC 0.743 -> below the 0.80 leakage-audit trigger
NOTE: wikileaks.org has exactly 30 users and scenario 1 has 30 insiders; a scenario-1 gain is expected to be easy here.


## 8. Descriptive: RF impurity importances for W2+E+H (fit on fold-0 training users only)

In [10]:
tr0, _ = week_masks(weekly, folds[0])
m = rf().fit(weekly.loc[tr0, SETS["W2+E+H"]], y[tr0])
imp = pd.Series(m.feature_importances_, index=SETS["W2+E+H"]).sort_values(ascending=False)
display(imp.head(20).to_frame("importance"))
print("share of total importance on named-site columns:", round(imp[[c for c in imp.index if any(k in c for k in ["_leak", "_job", "_cloud", "_download", "_competitor"])]].sum(), 3))

,importance
n_device,0.147
hist_mean_n_file,0.118
hist_mean_n_device,0.107
n_file,0.107
dev_n_device,0.063
hist_mean_wb_n_competitor,0.046
dev_n_file,0.042
dev_wb_n_competitor,0.028
hist_mean_wb_n_job,0.024
dev_wb_n_leak,0.020


share of total importance on named-site columns: 0.198


# Part 2 - POST-HOC audit and diagnostics (owner-approved after the first result)
Everything below is **post-hoc**: it never edits section 6b of the project log, feeds no feature, and changes no number above. Order set by the owner: audit (a-e) first, then diagnostics (f-i).
The >0.80 leakage-audit rule applies to ANY model: XGBoost W2+E+H = 0.879 and W2+H = 0.875 exceed it (the first report applied it to RF only).

In [11]:
import inspect, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from itgnn.comms_audit import (attack_week_position, audit_groups, draw_sample, flag_top_q, grouped_permutation_importance,
                               truncation_check, window_phase)
from itgnn.commslogs import WEB_CATEGORIES, match_suffix
from itgnn.windows import WEEK_FEATURE_COLS
from sklearn.metrics import average_precision_score
os.makedirs("results/figures", exist_ok=True)
assert windows["user"].is_unique
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "font.size": 10, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
def tick(label): print(f"[{label}] {time.time() - T0:.0f}s", file=LOG, flush=True)
T0 = time.time()

## 9a. Causality: the existing test, then an empirical truncation check

In [12]:
import re
txt = open("tests/test_weekly_features.py").read()
print(re.search(r"def test_causality_and_positive_control.*?(?=\n\ndef )", txt, re.S).group(0))
print()
print(re.search(r"    # causality: adding a week-2 event.*?\n\n\ndef", open("tests/test_commslogs.py").read(), re.S).group(0)[:400])

def test_causality_and_positive_control():
    cols = [c for s in FEATURE_SETS.values() for c in s]
    before = _build(_logon(BASE))
    # change data in week 6 and 7 only: earlier weeks must be identical
    later = BASE + [("U2", "PC1", 6, 9), ("U2", "PC7", 7, 9), ("U1", "PC8", 7, 9), ("U1", "PC1", 6, 20)]
    after = _build(_logon(later))
    key = ["user", "week_start"]
    b = before[before["week_start"] < MONDAYS[6]].set_index(key)[cols].sort_index()
    a = after[after["week_start"] < MONDAYS[6]].set_index(key)[cols].sort_index()
    pd.testing.assert_frame_equal(a, b)
    # positive control: the same kind of edit in the CURRENT week must change that week
    cur = _row(before, "U1", 6)
    new = _row(after, "U1", 6)
    assert new["n_logon"] != cur["n_logon"] and new["peers_mean"] != cur["peers_mean"]
    # and editing week 6 changes week 7's history but not week 6's history
    assert _row(after, "U1", 6)["hist_mean_n_logon"] == _row(before, "U1", 6)["hist_mean_n_logon"]


  

In [13]:
t = time.time()
CHECK_COLS = [c for c in SETS["W2+E+H"]]
sample = draw_sample(weekly, n=200, n_attack=30, n_weeks=15, seed=SEED)
print("sample:", len(sample), "user-weeks,", int(sample["label"].sum()), "attack weeks,", sample["week_start"].nunique(), "distinct calendar weeks,", sample["user"].nunique(), "users")
chk = truncation_check(sample, d["logon"], d["device"], d["file"], email, agg, full, CHECK_COLS)
print(f"truncated at the END of each week: rows compared {len(chk)}, max |diff| over {len(CHECK_COLS)} columns = {chk['max_abs_diff'].max()}, rows with a difference > 1e-9: {(chk['max_abs_diff'] > 1e-9).sum()} ({time.time() - t:.0f}s)")
# positive control: cutting INSIDE the week (3 days kept) must change the current-week features
ctrl = truncation_check(sample[sample["week_start"] == sample["week_start"].iloc[0]], d["logon"], d["device"], d["file"], email, agg, full, CHECK_COLS, cut_days=3)
print("positive control (week truncated after 3 days): rows that differ", int((ctrl['max_abs_diff'] > 1e-9).sum()), "of", len(ctrl), "| worst columns:", ctrl.loc[ctrl['max_abs_diff'] > 0, 'worst_col'].value_counts().head(5).to_dict())
assert chk["max_abs_diff"].max() < 1e-9 and (ctrl["max_abs_diff"] > 1e-9).any()
tick("9a")

sample: 200 user-weeks, 30 attack weeks, 15 distinct calendar weeks, 179 users


truncated at the END of each week: rows compared 200, max |diff| over 79 columns = 0.0, rows with a difference > 1e-9: 0 (60s)


positive control (week truncated after 3 days): rows that differ 18 of 18 | worst columns: {'em_size_sum': 18}


## 9b. Grouped permutation importance on held-out folds (PR-AUC drop, 5 shuffles; not impurity importance)

In [14]:
COLS = SETS["W2+E+H"]; GROUPS = audit_groups(COLS)
print({g: len(c) for g, c in GROUPS.items()}, "| 'history/deviation' overlaps the families (their own hist/dev columns are in both)")
perm_rows, base_pr = [], {}
for f in folds:
    tr, te = week_masks(weekly, f)
    for a, mk in ALGOS.items():
        m = mk(y[tr]).fit(weekly.loc[tr, COLS], y[tr])
        base_pr[(a, f["fold"])] = average_precision_score(y[te], m.predict_proba(weekly.loc[te, COLS])[:, 1])
        p = grouped_permutation_importance(m, weekly.loc[te, COLS], y[te], GROUPS, n_shuffles=5, seed=SEED + f["fold"])
        perm_rows.append(p.groupby("group")["drop"].mean().rename("drop").reset_index().assign(algo=a, fold=f["fold"]))
perm = pd.concat(perm_rows)
pt = perm.pivot_table(index="group", columns="algo", values="drop", aggfunc=["mean", "std"])
out = pd.DataFrame({a: pt[("mean", a)].map("{:.3f}".format) + " ± " + pt[("std", a)].map("{:.3f}".format) for a in ["RF", "XGB"]})
out["XGB sort"] = pt[("mean", "XGB")]; display(out.sort_values("XGB sort", ascending=False).drop(columns="XGB sort"))
print("PR-AUC drop = baseline PR-AUC minus PR-AUC with that group's columns permuted together on the held-out fold; mean ± std over 5 folds (each fold's value is the mean of 5 shuffles).")
tick("9b")

{'W2 base': 6, 'email volume': 12, 'email recipients/attachments': 15, 'web volume': 3, 'web domain counts': 3, 'web after-hours': 3, 'named-site': 18, 'first-seen': 9, 'history/deviation': 52} | 'history/deviation' overlaps the families (their own hist/dev columns are in both)


,RF,XGB
group,,
history/deviation,0.643 ± 0.094,0.701 ± 0.095
named-site,0.553 ± 0.089,0.627 ± 0.059
W2 base,0.258 ± 0.103,0.560 ± 0.129
web volume,0.047 ± 0.046,0.171 ± 0.037
email recipients/attachments,0.072 ± 0.051,0.075 ± 0.051
first-seen,0.031 ± 0.011,0.048 ± 0.005
web after-hours,0.026 ± 0.034,0.008 ± 0.015
email volume,0.001 ± 0.051,0.007 ± 0.022
web domain counts,-0.002 ± 0.008,0.005 ± 0.010


PR-AUC drop = baseline PR-AUC minus PR-AUC with that group's columns permuted together on the held-out fold; mean ± std over 5 folds (each fold's value is the mean of 5 shuffles).


## 9c. Leave-one-group-out refits (delta PR-AUC vs W2+E+H on the same fold)

In [15]:
logo = []
for f in folds:
    tr, te = week_masks(weekly, f)
    for a, mk in ALGOS.items():
        for g, gc in GROUPS.items():
            cols = [c for c in COLS if c not in set(gc)]
            s = mk(y[tr]).fit(weekly.loc[tr, cols], y[tr]).predict_proba(weekly.loc[te, cols])[:, 1]
            logo.append({"algo": a, "fold": f["fold"], "group": g, "delta": average_precision_score(y[te], s) - base_pr[(a, f["fold"])]})
logo = pd.DataFrame(logo)
lt = logo.groupby(["group", "algo"])["delta"].agg(mean="mean", std="std", folds_worse=lambda s: int((s < 0).sum())).unstack("algo")
lt.columns = [f"{stat} {algo}" for stat, algo in lt.columns]
display(lt)
print("delta = PR-AUC without the group minus PR-AUC of the full W2+E+H on the same fold (negative = the group helped). W2 reference: RF 0.499, XGB 0.465; full W2+E+H: RF 0.743, XGB 0.879.")
tick("9c")

,mean RF,mean XGB,std RF,std XGB,folds_worse RF,folds_worse XGB
group,,,,,,
W2 base,0.028,-0.003,0.020,0.011,0,3
email recipients/attachments,0.016,0.001,0.123,0.013,2,4
email volume,0.004,-0.002,0.052,0.010,3,3
first-seen,-0.033,-0.002,0.029,0.017,4,2
history/deviation,-0.157,-0.211,0.055,0.070,5,5
named-site,-0.068,-0.055,0.054,0.036,4,5
web after-hours,0.006,-0.003,0.028,0.017,2,3
web domain counts,0.019,-0.004,0.031,0.009,1,2
web volume,-0.031,-0.052,0.052,0.039,3,4


delta = PR-AUC without the group minus PR-AUC of the full W2+E+H on the same fold (negative = the group helped). W2 reference: RF 0.499, XGB 0.465; full W2+E+H: RF 0.743, XGB 0.879.


## 9d. Where does the model fire? (OOF scores, repeat 0)

In [16]:
def rank_pct(df): return df.assign(pct=df.groupby("fold")["score"].rank(pct=True))
FIRE = {}
rows_pos, rows_len, rows_ph = [], [], []
for m in ["RF W2", "RF W2+E+H", "XGB W2", "XGB W2+E+H"]:
    om = rank_pct(oof[oof["model"] == m].copy())
    om["flag"] = flag_top_q(om, 1, SEED)
    a = attack_week_position(om, windows)
    r = a.groupby("position")["flag"].agg(["mean", "size"]).add_prefix(f"{m}: ")
    rows_pos.append(r)
    rows_len.append(a.groupby(pd.cut(a["win_days"], [0, 7, 14, 30, 400], labels=["<=7 d", "8-14 d", "15-30 d", ">30 d"]), observed=True)["flag"].agg(["mean", "size"]).add_prefix(f"{m}: "))
    ph = window_phase(om, windows)
    t = ph.groupby("phase").agg(rows=("score", "size"), insiders=("user", "nunique"), mean_score=("score", "mean"), mean_pct=("pct", "mean")).loc[["before", "inside", "after"]]
    nr = om[~om["user"].isin(windows["user"])]
    t.loc["normal users (all weeks)"] = [len(nr), nr["user"].nunique(), nr["score"].mean(), nr["pct"].mean()]
    rows_ph.append(t.assign(model=m)); FIRE[m] = om
print("recall of attack WEEKS at q=1% (top 1% of test rows per fold), by position among the user's attack weeks"); display(pd.concat(rows_pos, axis=1).reindex(["first", "middle", "last", "only"]))
print("... by window length (days, start..end inclusive)"); display(pd.concat(rows_len, axis=1))
print("mean score and mean within-fold percentile rank; before = rows in the 28 days before the window start, after = 28 days after its end (rows exist only if the user is still active)")
ph_all = pd.concat(rows_ph); ph_all.index.name = "phase"; display(ph_all.reset_index().set_index(["model", "phase"]))
tick("9d")

recall of attack WEEKS at q=1% (top 1% of test rows per fold), by position among the user's attack weeks


,RF W2: mean,RF W2: size,RF W2+E+H: mean,RF W2+E+H: size,XGB W2: mean,XGB W2: size,XGB W2+E+H: mean,XGB W2+E+H: size
position,,,,,,,,
first,0.680,50,0.800,50,0.580,50,0.940,50
middle,0.550,202,0.866,202,0.564,202,0.926,202
last,0.700,50,0.880,50,0.640,50,0.920,50
only,0.700,20,0.850,20,0.600,20,0.900,20


... by window length (days, start..end inclusive)


,RF W2: mean,RF W2: size,RF W2+E+H: mean,RF W2+E+H: size,XGB W2: mean,XGB W2: size,XGB W2+E+H: mean,XGB W2+E+H: size
win_days,,,,,,,,
<=7 d,0.786,42,0.929,42,0.595,42,0.952,42
8-14 d,0.850,20,1.000,20,0.850,20,1.000,20
>30 d,0.554,260,0.835,260,0.558,260,0.915,260


mean score and mean within-fold percentile rank; before = rows in the 28 days before the window start, after = 28 days after its end (rows exist only if the user is still active)


rows  insiders  mean_score  mean_pct
model      phase                                                          
RF W2      before                      210        70       0.147     0.742
           inside                      322        70       0.453     0.976
           after                        91        42       0.040     0.810
           normal users (all weeks)  60362       930       0.003     0.489
RF W2+E+H  before                      210        70       0.055     0.699
           inside                      322        70       0.445     0.994
           after                        91        42       0.050     0.918
           normal users (all weeks)  60362       930       0.003     0.491
XGB W2     before                      210        70       0.389     0.723
           inside                      322        70       0.814     0.983
           after                        91        42       0.108     0.706
           normal users (all weeks)  60362       930       0.012     0.489
XGB W2+E+H before                      210        70       0.099     0.763
           inside                      322        70       0.886     0.996
           after                        91        42       0.096     0.634
           normal users (all weeks)  60362       930       0.003     0.490

## 9e. Is `wb_share_rare` exactly constant?

In [17]:
v = weekly["wb_share_rare"]
print("unique values:", v.nunique(), "| user-weeks > 0:", int((v > 0).sum()), "of", len(v), f"({(v > 0).mean():.2%}) | mean {v.mean():.4f} | max {v.max():.2f}")
print("=> not constant, so it is KEPT (the rule drops it only if exactly constant).")
print("by calendar-week block, share of user-weeks with wb_share_rare > 0:"); display(weekly.groupby(pd.cut(weekly["cal_week"], [-1, 7, 15, 30, 50, 80]), observed=True)["wb_share_rare"].agg(lambda s: (s > 0).mean()).to_frame("share > 0"))

unique values: 44 | user-weeks > 0: 114 of 63238 (0.18%) | mean 0.0000 | max 0.02
=> not constant, so it is KEPT (the rule drops it only if exactly constant).
by calendar-week block, share of user-weeks with wb_share_rare > 0:


,share > 0
cal_week,
"(-1, 7]",0.006
"(7, 15]",0.003
"(15, 30]",0.002
"(30, 50]",0.001
"(50, 80]",0.001


## 10f. POST-HOC: attack weeks and W2 features by calendar week (why does the calendar control lift?)

In [18]:
aw = weekly[weekly["label"] == 1].groupby("cal_week").size().reindex(range(weekly["cal_week"].min(), weekly["cal_week"].max() + 1), fill_value=0)
rows = weekly.groupby("cal_week").size().reindex(aw.index, fill_value=0)
blk = pd.cut(weekly["cal_week"], [-1, 9, 19, 29, 39, 49, 59, 80])
tab = weekly.groupby(blk, observed=True).agg(rows=("label", "size"), attack_weeks=("label", "sum"), prevalence=("label", "mean"),
        n_logon=("n_logon", "mean"), n_after_hours=("n_after_hours", "mean"), n_pcs=("n_pcs", "mean"), n_device=("n_device", "mean"), n_file=("n_file", "mean"), n_new_pcs=("n_new_pcs", "mean"))
display(tab)
print("Spearman correlation of each W2 base feature with calendar week (all rows):")
display(weekly[WEEK_FEATURE_COLS + ["n_new_pcs"]].corrwith(weekly["cal_week"], method="spearman").to_frame("rho"))
by_s = weekly[weekly["label"] == 1].assign(s=lambda x: x["user"].map(scen)).groupby("s")["cal_week"].agg(["min", "median", "max", "size"])
print("attack-week calendar weeks by scenario:"); display(by_s)
print(f"attack weeks: {int(aw.sum())}; weeks with at least one: {(aw > 0).sum()} of {len(aw)}; the busiest 10 calendar weeks hold {aw.nlargest(10).sum()} ({aw.nlargest(10).sum() / aw.sum():.0%})")

,rows,attack_weeks,prevalence,n_logon,n_after_hours,n_pcs,n_device,n_file,n_new_pcs
cal_week,,,,,,,,,
"(-1, 9]",5044,0,0.000,7.393,0.546,1.782,6.478,7.168,0.586
"(9, 19]",9827,0,0.000,7.206,0.541,1.764,6.226,6.770,0.496
"(19, 29]",9710,43,0.004,7.038,0.508,1.732,6.146,6.592,0.402
"(29, 39]",9493,106,0.011,7.105,0.479,1.685,6.344,6.855,0.321
"(39, 49]",9178,79,0.009,6.957,0.463,1.650,5.894,6.576,0.255
"(49, 59]",8851,56,0.006,6.776,0.397,1.571,5.726,6.370,0.192
"(59, 80]",11135,38,0.003,6.660,0.380,1.537,5.581,6.263,0.159


Spearman correlation of each W2 base feature with calendar week (all rows):


,rho
n_logon,-0.067
n_after_hours,-0.025
n_pcs,-0.035
n_device,-0.023
n_file,-0.022
n_new_pcs,-0.035


attack-week calendar weeks by scenario:


,min,median,max,size
s,,,,
1,27,39.000,61,52
2,24,42.000,67,260
3,23,35.500,69,10


attack weeks: 322; weeks with at least one: 46 of 69; the busiest 10 calendar weeks hold 115 (36%)


## 10g. POST-HOC: the stricter baseline W2+calendar, and W2+calendar+E+H

In [19]:
EXTRA = {"W2+calendar+E+H": SETS["W2+E+H"] + ["cal_week"]}
xr, xo = [], []
for f in folds:
    tr, te = week_masks(weekly, f)
    for sname, cols in EXTRA.items():
        for a, mk in ALGOS.items():
            s = mk(y[tr]).fit(weekly.loc[tr, cols], y[tr]).predict_proba(weekly.loc[te, cols])[:, 1]
            ev = evaluate_scores(y[te].to_numpy(), s, k=100); name = f"{a} {sname}"
            xr.append({"model": name, "algo": a, "set": sname, "repeat": 0, "fold": f["fold"], **ev})
            xo.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(), "cal_week": weekly.loc[te, "cal_week"].to_numpy(),
                                    "repeat": 0, "fold": f["fold"], "model": name, "score": s, "label": y[te].to_numpy()}))
results_all = pd.concat([results, pd.DataFrame(xr)], ignore_index=True); oof_all = pd.concat([oof] + xo, ignore_index=True)
oof_all.to_parquet("results/comms_probe_oof_scores.parquet", index=False)
pv2 = results_all.pivot_table(index="fold", columns="model", values="pr_auc")
rows = {}
for a in ["RF", "XGB"]:
    for hi, lo in [("W2+E+H", "W2+calendar"), ("W2+calendar+E+H", "W2+calendar"), ("W2+calendar+E+H", "W2+E+H"), ("W2+E+H generic", "W2+calendar")]:
        dl = pv2[f"{a} {hi}"] - pv2[f"{a} {lo}"]
        rows[f"{a}: {hi} - {lo}"] = {"mean diff": dl.mean(), "std": dl.std(), "folds positive": f"{int((dl > 0).sum())}/5", **{f"fold {i}": dl[i] for i in dl.index}}
display(pd.DataFrame(rows).T)
print("PR-AUC of W2+calendar+E+H:", {a: f"{pv2[f'{a} W2+calendar+E+H'].mean():.3f} ± {pv2[f'{a} W2+calendar+E+H'].std():.3f}" for a in ['RF', 'XGB']})
tick("10g")

,mean diff,std,folds positive,fold 0,fold 1,fold 2,fold 3,fold 4
RF: W2+E+H - W2+calendar,0.153,0.111,5/5,0.220,0.068,0.173,0.015,0.287
RF: W2+calendar+E+H - W2+calendar,0.164,0.084,5/5,0.216,0.126,0.189,0.038,0.252
RF: W2+calendar+E+H - W2+E+H,0.012,0.034,3/5,-0.004,0.058,0.016,0.023,-0.034
RF: W2+E+H generic - W2+calendar,0.085,0.122,4/5,0.179,0.013,0.174,-0.096,0.153
XGB: W2+E+H - W2+calendar,0.314,0.143,5/5,0.327,0.186,0.481,0.154,0.421
XGB: W2+calendar+E+H - W2+calendar,0.347,0.131,5/5,0.354,0.220,0.484,0.209,0.468
XGB: W2+calendar+E+H - W2+E+H,0.033,0.020,5/5,0.027,0.033,0.003,0.055,0.047
XGB: W2+E+H generic - W2+calendar,0.259,0.156,5/5,0.274,0.175,0.441,0.043,0.363


PR-AUC of W2+calendar+E+H: {'RF': '0.754 ± 0.105', 'XGB': '0.912 ± 0.065'}


## 10h/10i. POST-HOC: who visits the category domains (by scenario) - and the five excluded domains (REPORT ONLY, never in any model)

In [20]:
EXCLUDED = ["relytec.com", "wellresearchedreviews.com", "blackhatworld.com", "download.cnet.com", "custhelp.com"]
w = windows.set_index("user")
a2 = agg.assign(scenario=agg["user"].map(scen), insider=agg["user"].isin(labels))
st, en = a2["user"].map(w["start"]), a2["user"].map(w["end"])
a2["in_window"] = (a2["week_start"] <= en) & (a2["week_start"] + pd.Timedelta(days=7) > st)
def overlap(entries, name):
    m = a2[match_suffix(a2["domain"], entries)]
    v = m.groupby("user").agg(insider=("insider", "first"), scen=("scenario", "first"), win=("in_window", "any"), visits=("n", "sum"))
    r = {"visitors": len(v), "normal users": int((~v["insider"]).sum()), "insiders": int(v["insider"].sum())}
    for s in SCEN: r[f"s{s} insiders"] = int((v["scen"] == s).sum())
    r["insiders visiting inside their window"] = int(v.loc[v["insider"], "win"].sum())
    r["visits"] = int(v["visits"].sum())
    return pd.Series(r, name=name)
ins_by_s = scen.value_counts().sort_index().to_dict(); print("insiders per scenario:", ins_by_s, "| normal users 930")
display(pd.DataFrame([overlap(v, k) for k, v in WEB_CATEGORIES.items()]))
print("per listed domain:"); display(pd.DataFrame([overlap([dm], dm) for v in WEB_CATEGORIES.values() for dm in v]).loc[lambda x: ~x.index.duplicated()])
print("EXCLUDED domains (report only):"); display(pd.DataFrame([overlap([dm], dm) for dm in EXCLUDED]))

insiders per scenario: {1: 30, 2: 30, 3: 10} | normal users 930


,visitors,normal users,insiders,s1 insiders,s2 insiders,s3 insiders,insiders visiting inside their window,visits
leak,30,0,30,30,0,0,30,69
job,906,839,67,27,30,10,59,413952
cloud,859,798,61,24,28,9,46,582104
download,9,0,9,0,0,9,9,34
competitor,513,463,50,15,30,5,49,218201


per listed domain:


,visitors,normal users,insiders,s1 insiders,s2 insiders,s3 insiders,insiders visiting inside their window,visits
wikileaks.org,30,0,30,30,0,0,30,69
careerbuilder.com,799,737,62,23,30,9,52,117221
indeed.com,560,507,53,16,30,7,48,45021
monster.com,556,504,52,16,30,6,46,54447
simplyhired.com,591,538,53,18,30,5,49,52833
job-hunt.org,512,462,50,15,30,5,47,43611
jobhuntersbible.com,508,458,50,15,30,5,44,43463
taleo.net,513,472,41,18,15,8,22,57356
dropbox.com,290,267,23,10,9,4,16,41128
mediafire.com,537,503,34,16,16,2,26,331283


EXCLUDED domains (report only):


,visitors,normal users,insiders,s1 insiders,s2 insiders,s3 insiders,insiders visiting inside their window,visits
relytec.com,5,0,5,0,0,5,5,5
wellresearchedreviews.com,5,0,5,0,0,5,5,6
blackhatworld.com,5,5,0,0,0,0,0,8
download.cnet.com,5,0,5,0,0,5,5,6
custhelp.com,5,5,0,0,0,0,0,126


## 11. Why does RF W2+E+H catch fewer insiders than RF W2 at q = 0.25%? (repeat 0)

In [21]:
rows = []
for m in ["RF W2", "RF W2+E+H", "XGB W2", "XGB W2+E+H"]:
    om = FIRE.get(m, rank_pct(oof[oof["model"] == m].copy())); fl = flag_top_q(om, 0.25, SEED); g = om[fl]
    att = g[g["label"] == 1]; per = att.groupby("user").size().sort_values(ascending=False)
    rows.append({"model": m, "weeks flagged": len(g), "attack weeks flagged": len(att), "precision": len(att) / len(g), "distinct insiders": att["user"].nunique(),
                 "max flagged weeks / insider": int(per.max()), "insiders with >=3 flagged weeks": int((per >= 3).sum()),
                 "share of attack weeks from top-5 insiders": per.head(5).sum() / len(att), "normal flagged": g.loc[~g["user"].isin(labels), "user"].nunique()})
display(pd.DataFrame(rows).set_index("model"))
print("If precision rises while distinct insiders fall, the stronger model piles its few alert slots onto the same insiders' consecutive attack weeks.")
tick("11")

,weeks flagged,attack weeks flagged,precision,distinct insiders,max flagged weeks / insider,insiders with >=3 flagged weeks,share of attack weeks from top-5 insiders,normal flagged
model,,,,,,,,
RF W2,161,111,0.689,49,6,20,0.216,3
RF W2+E+H,161,138,0.857,41,9,15,0.312,2
XGB W2,161,103,0.640,51,7,14,0.243,2
XGB W2+E+H,161,159,0.988,56,8,19,0.233,0


If precision rises while distinct insiders fall, the stronger model piles its few alert slots onto the same insiders' consecutive attack weeks.


## 12. Charts

In [22]:
order_sets = list(SETS) + list(EXTRA)
fig, ax = plt.subplots(figsize=(10.4, 5.2))
ypos = {s: i for i, s in enumerate(order_sets[::-1])}
for a, col, off in [("RF", BLUE, 0.17), ("XGB", ORANGE, -0.17)]:
    for s in order_sets:
        v = pv2[f"{a} {s}"]
        ax.scatter(v.values, [ypos[s] + off] * len(v), s=36, color=col, alpha=0.85, edgecolor="#fcfcfb", linewidth=1.2, zorder=3)
        ax.plot([v.mean()] * 2, [ypos[s] + off - 0.13, ypos[s] + off + 0.13], color=col, lw=2.2, zorder=4)
ax.set_yticks(list(ypos.values())); ax.set_yticklabels(list(ypos.keys())); ax.set_xlabel("PR-AUC per test fold (dots) and mean (bar)")
ax.set_title("Per-fold PR-AUC by feature set, repeat 0 (5 folds)", loc="left", fontsize=11, color=INK)
ax.scatter([], [], color=BLUE, s=36, label="Random forest"); ax.scatter([], [], color=ORANGE, s=36, label="XGBoost")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False); ax.grid(axis="y", visible=False); fig.tight_layout(); fig.savefig("results/figures/06_pr_auc_by_set.png", dpi=130, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(8.6, 3.6))
ax.bar(aw.index, aw.values, color=BLUE, width=0.8)
ax.set_xlabel("calendar week (0 = week of 2010-01-04)"); ax.set_ylabel("attack weeks (labelled user-weeks)")
ax.set_title(f"Attack weeks by calendar week ({int(aw.sum())} in total; rows after the warm-up)", loc="left", fontsize=11, color=INK)
ax.grid(axis="x", visible=False); fig.tight_layout(); fig.savefig("results/figures/06_attack_weeks_by_calendar_week.png", dpi=130); plt.close(fig)

pm = perm.groupby(["group", "algo"])["drop"].agg(["mean", "std"]).unstack("algo")
order_g = pm[("mean", "XGB")].sort_values().index
fig, ax = plt.subplots(figsize=(8.6, 4.8)); yy = np.arange(len(order_g))
for a, col, off in [("RF", BLUE, 0.2), ("XGB", ORANGE, -0.2)]:
    ax.barh(yy + off, pm.loc[order_g, ("mean", a)], height=0.36, color=col, label={"RF": "Random forest", "XGB": "XGBoost"}[a],
            xerr=pm.loc[order_g, ("std", a)], error_kw={"elinewidth": 1, "ecolor": MUTED, "capsize": 2})
ax.set_yticks(yy); ax.set_yticklabels(order_g); ax.set_xlabel("PR-AUC drop when the group is permuted (held-out folds; bar = fold std)")
ax.set_title("Grouped permutation importance, W2+E+H, repeat 0", loc="left", fontsize=11, color=INK)
ax.legend(loc="lower right", frameon=False); ax.grid(axis="y", visible=False); fig.tight_layout(); fig.savefig("results/figures/06_grouped_permutation_importance.png", dpi=130); plt.close(fig)
print("saved results/figures/06_*.png"); tick("12")

saved results/figures/06_*.png
